# Lab 10 — Containerizing an OpenAI-Compatible API

**Production Readiness Pack | Docker on your own machine | `OPENAI_API_KEY` to run the container**

---

The Lab 5 server ran in Colab, started by `subprocess.Popen`, reached through ngrok. None of that exists anywhere else. Ask a platform team to run it and the first thing they will say is: send us an image.

An image is your server plus everything it needs to start, frozen into one file that runs the same way on your laptop, in a CI job and on a cloud service. This lab writes the four files that make one, then builds it and calls it with the same OpenAI client as always.

**Colab can write the files but cannot build them.** Colab has no Docker daemon. Run the notebook anywhere, then do the build on a machine with Docker installed. (Checked 2026-09-23 on Docker 29: the image builds to about 205 MB and runs as a non-root user.)

**Coming from Lab 5 (and Lab 9):** same routes, same wire format. What changes is how the server gets started, and where its settings come from.

## What a platform team is really asking

"Can we have an image?" is short for a list of questions:

- Which Python, which packages, which versions?
- What command starts it, and on which port?
- Where do the secrets come from?
- How do we know it is healthy, so we can restart it when it is not?
- Can the same image run in staging and production with different settings?

The four files below answer all of them. Notice that the LLM is barely involved. The deployment unit has to behave like any other piece of software.

---

## 1. The server

`server.py` in three cells, the same shape as Lab 5. The difference is at the top: every setting comes from an environment variable. That is the contract between the image and whoever runs it.

In [ ]:
%%writefile server.py
import os
from typing import Any, Dict, List, Optional

from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from openai import OpenAI
from pydantic import BaseModel, Field

# Everything configurable comes from the environment. That is the container contract.
MODEL_NAME = os.getenv("MODEL_NAME", "gpt-4o-mini")
UPSTREAM_BASE_URL = os.getenv("UPSTREAM_BASE_URL")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

app = FastAPI(title="OpenAI-Compatible LLM API", version="1.0.0")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: Optional[str] = None
    messages: List[ChatMessage]
    temperature: float = 0.2
    stream: bool = False
    max_tokens: Optional[int] = Field(default=None, ge=1)

The upstream client is built per request, so a missing key becomes a clean `500` with a message instead of a crash at startup. (Run the container without a key and that is exactly what you get.) `/health` and `/v1/models` are what an orchestrator probes.

In [ ]:
%%writefile -a server.py

def upstream_client() -> OpenAI:
    if not OPENAI_API_KEY:
        raise HTTPException(status_code=500, detail="OPENAI_API_KEY is not configured")
    kwargs: Dict[str, Any] = {"api_key": OPENAI_API_KEY}
    if UPSTREAM_BASE_URL:
        kwargs["base_url"] = UPSTREAM_BASE_URL
    return OpenAI(**kwargs)

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME, "upstream_base_url": UPSTREAM_BASE_URL or "https://api.openai.com/v1"}

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": MODEL_NAME, "object": "model", "owned_by": "classroom-api"}]}

The chat route is Lab 5's, with one simplification: streamed chunks from the upstream are passed straight through with `chunk.model_dump_json()` instead of being rebuilt.

---

## 2. Dependencies and exclusions

`requirements.txt` pins the server's exact versions, so every build installs the same thing. `.dockerignore` lists what must never be copied into the image. The `.env` you use to run these labs locally is first on that list: an image with a key inside is a leaked key the moment anyone pulls it.

In [ ]:
%%writefile -a server.py

@app.post("/v1/chat/completions")
def chat_completions(request: ChatCompletionRequest):
    client = upstream_client()
    model = request.model or MODEL_NAME
    messages = [m.model_dump() for m in request.messages]

    if request.stream:
        def event_stream():
            stream = client.chat.completions.create(model=model, messages=messages, temperature=request.temperature,
                                                    max_tokens=request.max_tokens, stream=True)
            for chunk in stream:
                yield "data: " + chunk.model_dump_json() + "\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(event_stream(), media_type="text/event-stream")

    response = client.chat.completions.create(model=model, messages=messages, temperature=request.temperature,
                                              max_tokens=request.max_tokens)
    return response.model_dump()

In [ ]:
%%writefile requirements.txt
fastapi==0.115.6
uvicorn[standard]==0.34.0
openai>=1.57.0
pydantic>=2.10.0

In [ ]:
%%writefile .dockerignore
.env
.env.*
__pycache__/
*.pyc
*.ipynb
.ipynb_checkpoints/
chroma_db/
vector_db/
mlflow.db
mlruns/
.git/
.DS_Store

## What goes in the image, and what does not

The image holds code and dependencies. Everything that differs between environments, or that is secret, arrives when the container starts.

| Built into the image | Given to the container at runtime |
| --- | --- |
| `server.py` | `OPENAI_API_KEY` |
| Python packages | `MODEL_NAME` |
| The start command | `UPSTREAM_BASE_URL` |
| Safe defaults | Client keys, tenant settings |

That split is what lets one image talk to OpenAI today and to your own vLLM server tomorrow: change `UPSTREAM_BASE_URL`, restart, same image.

---

## 3. The Dockerfile

Start from a slim Python image, install the pinned packages, copy the server in, and switch to a normal user before starting. Running as root inside a container is the default and a bad one: if the server is ever compromised, the attacker is root.

`--host 0.0.0.0` is the one difference from the command Lab 5 used. Traffic reaches a container from outside it, so the server has to listen on every interface, not just `127.0.0.1`.

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

ENV PYTHONDONTWRITEBYTECODE=1     PYTHONUNBUFFERED=1

WORKDIR /app

RUN useradd --create-home --shell /bin/bash appuser

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY server.py .

USER appuser
EXPOSE 8000

CMD ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"]

## 4. Check the files

The cell below just confirms all four exist. Before any Docker, you can also run the server directly on your machine to make sure it works:

```bash
export OPENAI_API_KEY=your_key
uvicorn server:app --port 8000
```

In [ ]:
from pathlib import Path
for file_name in ["server.py", "requirements.txt", ".dockerignore", "Dockerfile"]:
    path = Path(file_name)
    print(f"{file_name}: {path.stat().st_size} bytes")

---

## 5. Build and run

On a machine with Docker, in the folder with the four files:

```bash
docker build -t llm-api:lab10 .
docker run --rm -p 8000:8000 -e OPENAI_API_KEY="$OPENAI_API_KEY" -e MODEL_NAME="gpt-4o-mini" llm-api:lab10
```

The key goes in with `-e` at run time. It is never in the image.

To send requests somewhere other than OpenAI (vLLM, Ollama, LiteLLM, any OpenAI-compatible server), add one more flag:

```bash
-e UPSTREAM_BASE_URL="https://your-upstream.example.com/v1"
```

Two quick checks once it is up:

```bash
curl localhost:8000/health                  # {"status":"ok", ...}
docker run --rm llm-api:lab10 whoami        # appuser, not root
```

In [ ]:
# Optional local check. In Colab this often prints that Docker is unavailable.
!docker --version || true

## 6. Call it with the OpenAI client

The client has no idea there is a container, or FastAPI, behind the URL. Run this on the machine where the container is running:

```python
from openai import OpenAI

client = OpenAI(api_key="not-used-by-local-proxy", base_url="http://localhost:8000/v1")
print(client.models.list())

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say hello from the containerized API."}],
)
print(response.choices[0].message.content)
```

---

## What a container does not solve

- **It does not make inference scale.** This image forwards requests to someone else's GPUs. Running a model yourself still needs vLLM or similar (Bonus 03).
- **Its disk is temporary.** A Chroma folder written inside the container disappears on restart. Mount a volume or use a managed vector database.
- **Health checks must be cheap.** `/health` here never calls the model. If it did, every probe would cost money, and an OpenAI outage would get your healthy server restarted in a loop.
- **Logs go to stdout.** The platform collects them from there; files inside the container vanish with it.
- **One container serves many requests at once.** Shared global state that is not thread-safe will bite you.
- **Secrets come from the platform:** Secret Manager, AWS Secrets Manager, Kubernetes Secrets. Never the image.

## Where the image can go

| Platform | Good for | Notes |
| --- | --- | --- |
| Google Cloud Run | Simple stateless HTTP services | Scales to zero; the easiest first target |
| AWS App Runner | The same, on AWS | Less setup than ECS |
| ECS / Fargate | Production services on AWS | More control over networking and IAM |
| Kubernetes | Platforms running many services | Powerful, and a lot to operate |

All of them take this exact image. What changes is how you pass `OPENAI_API_KEY` and how many copies run.

---

## Try it

1. Add an `X-Request-ID` header to every response (hint: a FastAPI middleware). Why would Lab 8's traces want it?
2. Make the server require its own API key from callers, read from an environment variable. Test it with the wrong key.
3. Rebuild with a change to `server.py` only and watch which build steps Docker reuses. Why does `COPY requirements.txt` come before `COPY server.py`?
4. For your Capstone, which would you choose: a Hugging Face Space, this container, or a vLLM server? Why?

## What to take with you

1. **An image is the unit a platform can run.** Code, packages and the start command, frozen together.
2. **Settings and secrets arrive at runtime.** The same image runs anywhere; only its environment changes.
3. **Run as a normal user and keep `/health` cheap.** Two defaults that save real incidents.
4. **A container packages your server; it does not scale a model.** That is vLLM's job.

## Next

[Lab 11 — Guardrails](../11_Guardrails_Security/README.md): Lab 7's red-team, turned into input, retrieval and output checks. No API key.